In [0]:
import time
from pyspark.sql import functions as F

results = []

# 1. Bronze append timing
start = time.time()

# put your bronze incremental load function call here
# Example:
# for table_name, file_name in file_names.items():
#     append_incremental_run(table_name, file_name, run_id, schema_name)

bronze_time = time.time() - start
results.append(("bronze_incremental_load", bronze_time))

# 2. Silver rebuild timing
start = time.time()

# run your silver notebook logic here or key transformation block

silver_time = time.time() - start
results.append(("silver_rebuild", silver_time))

# 3. Gold rebuild timing
start = time.time()

# run your gold notebook logic here or key transformation block

gold_time = time.time() - start
results.append(("gold_rebuild", gold_time))

total_time = bronze_time + silver_time + gold_time
results.append(("total_pipeline_time", total_time))

results_df = spark.createDataFrame(results, ["step_name", "execution_time_seconds"])
display(results_df)

step_name,execution_time_seconds
bronze_incremental_load,4.935264587402344E-5
silver_rebuild,4.506111145019531E-5
gold_rebuild,3.790855407714844E-5
total_pipeline_time,1.323223114013672E-4


Experiment 2: Raw vs Gold query benchmark

one business question.

Good example:
Which providers generated the highest total revenue?

Raw-style query timing

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()

raw_result = (
    spark.table("healthcare_project.bronze_encounters")
    .groupBy("PROVIDER")
    .agg(F.sum("TOTAL_CLAIM_COST").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

raw_result.count()

raw_query_time = time.time() - start
print("Raw query time:", raw_query_time)

Raw query time: 2.0046908855438232


In [0]:
start = time.time()

gold_result = (
    spark.table("healthcare_project.gold_provider_rankings")
    .select("provider_id", "provider_name", "total_revenue")
    .orderBy(F.desc("total_revenue"))
)

gold_result.count()

gold_query_time = time.time() - start
print("Gold query time:", gold_query_time)

Gold query time: 0.9075753688812256


In [0]:
improvement_pct = ((raw_query_time - gold_query_time) / raw_query_time) * 100
print(f"Query time improvement: {improvement_pct:.2f}%")

Query time improvement: 54.73%


In [0]:
run1 = 1159
run2 = 11577

growth = ((run2 - run1) / run1) * 100
print(f"Data volume increased by {growth:.2f}%")

Data volume increased by 898.88%


In [0]:
tables = [
    "bronze_patients",
    "bronze_encounters",
    "bronze_conditions",
    "bronze_procedures",
    "bronze_medications",
    "bronze_providers"
]

total_rows = 0

for table in tables:
    count = spark.table(f"healthcare_project.{table}").count()
    print(f"{table}: {count}")
    total_rows += count

print(f"\nTotal Bronze Rows Processed: {total_rows}")

bronze_patients: 12736
bronze_encounters: 742364
bronze_conditions: 461418
bronze_procedures: 2065888
bronze_medications: 623172
bronze_providers: 1963

Total Bronze Rows Processed: 3907541
